# Air Pollution in Algeria

## Setup

In [1]:
import warnings

import altair as alt
import altairtheme
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
altairtheme.enable()

In [2]:
from pathlib import Path


def find_project_root(marker="pyproject.toml"):
    """Walk up from this notebook's directory until we find the project root."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/farhanreynaldo/Documents/world-bank/git-repo/algeria-economic-monitoring


In [3]:
def clean_names(df):
    """Standardize column names: lowercase, strip, replace spaces with underscores."""
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    return df


def standardize_series(series):
    """Z-score standardization for comparable time-series overlay."""
    return (series - series.mean()) / series.std()


def fit_sector_models(
    df: pd.DataFrame, var: str | list[str], outcome: str = "gdp_log"
) -> list[tuple[str, object]]:
    """OLS: outcome ~ var for each economic_activity sector.

    Returns list of (sector_name, fitted_model).
    """
    var_str = " + ".join(var) if isinstance(var, list) else var
    model_list = []
    for sector in sorted(df["sector"].unique()):
        sector_df = df.query("sector == @sector").dropna(
            subset=[outcome] + (var if isinstance(var, list) else [var])
        )
        mod = smf.ols(f"{outcome} ~ {var_str}", data=sector_df).fit()
        model_list.append((sector, mod))
    return model_list


def plot_coefficients(
    model_list: list[tuple[str, object]],
    var: str | list[str] = None,
    title: str = "NO₂ Elasticity of GDP by Sector",
    subtitle: str | None = None,
) -> alt.LayerChart | alt.FacetChart:
    """Plot OLS coefficients with 95% CI error bars from sector models."""
    rows = []
    for activity, mod in model_list:
        params = mod.params.drop("const", errors="ignore")
        conf = mod.conf_int().drop("const", errors="ignore")
        pvals = mod.pvalues.drop("const", errors="ignore")
        for v in params.index:
            if var is not None:
                vars_list = [var] if isinstance(var, str) else list(var)
                if v not in vars_list:
                    continue
            rows.append(
                {
                    "economic_activity": activity,
                    "variable": v,
                    "coefficient": params[v],
                    "std_error": mod.bse[v],
                    "lower": conf.loc[v, 0],
                    "upper": conf.loc[v, 1],
                    "p_value": pvals[v],
                    "significant": pvals[v] < 0.05,
                }
            )

    coef_df = pd.DataFrame(rows)
    if coef_df.empty:
        print("No coefficients to plot.")
        return None

    vars_list = (
        [var]
        if isinstance(var, str)
        else list(var)
        if var is not None
        else coef_df["variable"].unique().tolist()
    )

    base = alt.Chart(coef_df).encode(
        y=alt.Y(
            "economic_activity:N",
            title="",
            sort=alt.EncodingSortField(field="coefficient", order="descending"),
        ),
    )

    points = base.mark_point(filled=True, size=60).encode(
        x=alt.X("coefficient:Q", title="Coefficient"),
        color=alt.condition(
            alt.datum.significant, alt.value("#E3120B"), alt.value("#AAAAAA")
        ),
        tooltip=[
            alt.Tooltip("economic_activity:N", title="Sector"),
            alt.Tooltip("variable:N", title="Variable"),
            alt.Tooltip("coefficient:Q", title="Coefficient", format=".4f"),
            alt.Tooltip("std_error:Q", title="Std. Error", format=".4f"),
            alt.Tooltip("lower:Q", title="Lower CI", format=".4f"),
            alt.Tooltip("upper:Q", title="Upper CI", format=".4f"),
            alt.Tooltip("p_value:Q", title="P-value", format=".4f"),
        ],
    )

    error_bars = base.mark_rule().encode(
        x=alt.X("lower:Q", title=""),
        x2="upper:Q",
        color=alt.condition(
            alt.datum.significant, alt.value("#E3120B"), alt.value("#AAAAAA")
        ),
    )

    zero_line = (
        alt.Chart(pd.DataFrame({"x": [0]}))
        .mark_rule(strokeDash=[4, 4], color="gray")
        .encode(x="x:Q")
    )

    if subtitle is None:
        var_label = " + ".join(vars_list)
        subtitle = f"OLS: gdp_log ~ {var_label} with 95% CI"

    layer = (points + error_bars + zero_line).properties(
        width=500,
        height=300,
        title=alt.Title(text=title, subtitle=subtitle),
    )

    if len(vars_list) > 1:
        return layer.facet(
            facet=alt.Facet("variable:N", title="Variable", sort=vars_list),
            columns=2,
        )
    return layer

## Data Loading & Preprocessing

### Quarterly GDP Sectoral Data

We extract two levels:
- **Detailed sectors** (rows 3–43): 40 individual sectors for the regression analysis
- **Aggregate sectors** (rows 44–51): 8 broad groupings for the regression analysis
- **Summary rows**: Total GDP (row 33), Hydrocarbons (row 41), Non-HC GDP (row 42)

In [4]:
gdp_file = DATA_DIR / "gdp" / "Rebased GDP 2026 DZA MTI.xlsx"
gdp_raw = pd.read_excel(gdp_file, sheet_name="Prod Con Qly", header=None)

# Row 0 = years (merged across 4 cols), Row 1 = quarters (T1–T4)
years_row = gdp_raw.iloc[0, 2:].ffill().values  # forward-fill merged year cells
quarters_row = gdp_raw.iloc[1, 2:].values

# Build date index from year + quarter
quarter_to_month = {"T1": 1, "T2": 4, "T3": 7, "T4": 10}
dates = []
for y, q in zip(years_row, quarters_row):
    if pd.notna(y) and pd.notna(q) and str(q).strip() in quarter_to_month:
        dates.append(
            pd.Timestamp(year=int(y), month=quarter_to_month[str(q).strip()], day=1)
        )
    else:
        dates.append(pd.NaT)

# Extract sector data (col 0 = English name, col 1 = French name, cols 2+ = values)
sector_names = gdp_raw.iloc[:, 0]
data_block = gdp_raw.iloc[:, 2:]
data_block.columns = dates


DETAILED_SECTOR_ROWS = {
    # 2: "Agriculture and fishing",
    3: "Shadow industry",
    4: "Oil and gas extraction and related services",
    5: "Other mining and quarrying",
    6: "Food and tobacco industries",
    7: "Textile, clothing and fur industries",
    8: "Leather and footwear industries",
    9: "Manufacture of wood and paper products, printing and reproduction",
    10: "Refining and coking",
    11: "Chemical, rubber and plastics industries",
    12: "Manufacture of other non-metallic mineral products",
    13: "Metallurgy, metalworking",
    14: "Manufacture of machinery and equipment",
    15: "Manufacture of office machinery and computers",
    16: "Manufacture of electrical machinery and apparatus",
    17: "Manufacture of communications equipment and medical instruments",
    18: "Other manufacturing",
    19: "Electricity and gas supply",
    # 20: "Construction",
    21: "Trade, repair of motor vehicles and personal and household goods",
    22: "Hotels and restaurants",
    23: "Transport and communications",
    24: "Financial activities",
    25: "Real estate, rental and business services",
    26: "Public administration",
    27: "Education, health and social work",
    28: "Community and social services",
}

# Aggregate sectors (0-indexed rows 43–50 = Excel rows 44–51)
AGGREGATE_ROWS = {
    43: "Agriculture and fishing",
    44: "Mining and refining",
    45: "Non-HC manufacturing",
    46: "Electricity and gas",
    47: "Construction",
    48: "Low VA services",
    49: "High VA services",
    50: "Non-commercial services",
}

# Summary rows
SUMMARY_ROWS = {
    32: "Total GDP",
    40: "Hydrocarbons",
    41: "Non-HC GDP",
}

all_rows = {**DETAILED_SECTOR_ROWS, **AGGREGATE_ROWS, **SUMMARY_ROWS}

gdp_frames = []
for row_idx, label in all_rows.items():
    series = data_block.iloc[row_idx].dropna()
    temp = pd.DataFrame({"date": series.index, "gdp": series.values, "sector": label})
    gdp_frames.append(temp)

gdp_all = (
    pd.concat(gdp_frames, ignore_index=True)
    .assign(
        date=lambda df: pd.to_datetime(df["date"]),
        gdp=lambda df: pd.to_numeric(df["gdp"], errors="coerce"),
        gdp_log=lambda df: np.log(df["gdp"]),
        quarter=lambda df: df["date"].dt.quarter,
    )
    .dropna(subset=["gdp"])
)

### National Monthly NO₂ Data

In [5]:
no2_national = (
    pd.read_csv(DATA_DIR / "airpollution" / "no2" / "no2_nasa_monthly_national.csv")
    .pipe(clean_names)
    .assign(
        date=lambda df: pd.to_datetime(df["date"]),
        log_no2=lambda df: np.log(df["no2_trop"]),
        log_no2_cloud_screened=lambda df: np.log(df["no2_trop_cloud_screened"]),
    )
).filter(
    [
        "date",
        "year",
        "month",
        "no2_trop",
        "no2_trop_cloud_screened",
        "log_no2",
        "log_no2_cloud_screened",
    ]
)

no2_national.head()

,date,year,month,no2_trop,no2_trop_cloud_screened,log_no2,log_no2_cloud_screened
0,2012-01-01,2012,1,0.000006,0.000006,-12.011003,-11.991250
1,2012-02-01,2012,2,0.000008,0.000008,-11.780784,-11.753597
2,2012-03-01,2012,3,0.000006,0.000007,-11.973511,-11.901845
3,2012-04-01,2012,4,0.000006,0.000007,-11.967331,-11.871617
4,2012-05-01,2012,5,0.000007,0.000008,-11.835056,-11.796386


In [6]:
no2_quarterly = (
    no2_national.set_index("date")
    .resample("QS")
    .agg(
        no2_trop=("no2_trop_cloud_screened", "mean"),
        no2_trop_total=("no2_trop", "mean"),
        log_no2=("log_no2_cloud_screened", "mean"),
        log_no2_total=("log_no2", "mean"),
    )
    .reset_index()
    .assign(
        quarter=lambda df: df["date"].dt.quarter,
        log_no2_lag_1=lambda df: df["log_no2"].shift(1),
        log_no2_lag_2=lambda df: df["log_no2"].shift(2),
        log_no2_lag_3=lambda df: df["log_no2"].shift(3),
    )
)
no2_quarterly.head()

,date,no2_trop,no2_trop_total,log_no2,log_no2_total,quarter,log_no2_lag_1,log_no2_lag_2,log_no2_lag_3
0,2012-01-01,0.000007,0.000007,-11.882231,-11.921766,1,NaN,NaN,NaN
1,2012-04-01,0.000007,0.000007,-11.863194,-11.920607,2,-11.882231,NaN,NaN
2,2012-07-01,0.000008,0.000008,-11.730008,-11.761442,3,-11.863194,-11.882231,NaN
3,2012-10-01,0.000007,0.000006,-11.943599,-11.983659,4,-11.730008,-11.863194,-11.882231
4,2013-01-01,0.000007,0.000006,-11.925341,-11.948862,1,-11.943599,-11.730008,-11.863194


### Provincial (ADM1) NO₂ Data

In [7]:
no2_adm1 = (
    pd.read_csv(DATA_DIR / "airpollution" / "no2" / "no2_nasa_monthly_adm1.csv")
    .pipe(clean_names)
    .assign(date=lambda df: pd.to_datetime(df["date"]))
)

no2_adm1_quarterly = (
    no2_adm1.groupby(["name_1", pd.Grouper(key="date", freq="QS")])
    .agg(
        no2_trop=("no2_trop", "mean"),
        no2_trop_cloud_screened=("no2_trop_cloud_screened", "mean"),
    )
    .assign(
        log_no2_trop=lambda df: np.log(df["no2_trop"]),
        log_no2_trop_cloud_screened=lambda df: np.log(df["no2_trop_cloud_screened"]),
    )
    .reset_index()
)

# INDUSTRIAL_WILAYAS = ["Alger", "Oran", "Ouargla", "Skikda"]
INDUSTRIAL_WILAYAS = ["Adrar", "Alger", "Constantine", "Blida"]
no2_industrial = no2_adm1.copy().query("name_1 in @INDUSTRIAL_WILAYAS")

no2_ind_quarterly = (
    no2_industrial.set_index("date")
    .resample("QS")
    .agg(
        no2_trop_industrial=("no2_trop", "mean"),
        no2_trop_cloud_screened_industrial=("no2_trop_cloud_screened", "mean"),
    )
    .assign(
        log_no2_industrial=lambda df: np.log(df["no2_trop_industrial"]),
        log_no2_cloud_screened_industrial=lambda df: np.log(
            df["no2_trop_cloud_screened_industrial"]
        ),
    )
    .reset_index()
)

no2_ind_quarterly.head()

,date,no2_trop_industrial,no2_trop_cloud_screened_industrial,log_no2_industrial,log_no2_cloud_screened_industrial
0,2012-01-01,0.000009,0.000010,-11.638806,-11.487140
1,2012-04-01,0.000007,0.000008,-11.834444,-11.724402
2,2012-07-01,0.000009,0.000010,-11.591982,-11.521165
3,2012-10-01,0.000008,0.000009,-11.731359,-11.583684
4,2013-01-01,0.000007,0.000008,-11.885186,-11.773224


### NO₂ Flaring Grids (Monthly, 2012–2025)

The monthly flaring grid data provides spatially granular NO₂ readings at known gas flaring locations across 2012–2025. We aggregate these grids to the national level.

In [8]:
no2_flaring_monthly = (
    pd.read_csv(
        DATA_DIR
        / "airpollution"
        / "no2"
        / "processed"
        / "no2_nasa_flaring_grids_2012_2025.csv"
    )
    .pipe(clean_names)
    .rename(columns={"flaring_site_count_x": "flaring_site_count"})
    .assign(date=lambda df: pd.to_datetime(df["date"]))
)

# Aggregate to national level per month
no2_flaring_national_monthly = (
    no2_flaring_monthly.groupby("date")
    .agg(
        no2_flaring_sum=("no2_trop", "sum"),
        no2_flaring_mean=("no2_trop", "mean"),
        total_flaring_sites=("flaring_site_count", "sum"),
        # Weighted mean: weight each grid's NO2 by its flaring_site_count
        no2_weighted_num=(
            "no2_trop",
            lambda x: (
                x * no2_flaring_monthly.loc[x.index, "flaring_site_count"]
            ).sum(),
        ),
        no2_weighted_den=("flaring_site_count", "sum"),
    )
    .assign(
        no2_flaring_weighted=lambda df: df["no2_weighted_num"] / df["no2_weighted_den"]
    )
    .drop(columns=["no2_weighted_num", "no2_weighted_den"])
    .reset_index()
)

# Convert to quarterly
no2_flaring_quarterly = (
    no2_flaring_national_monthly.set_index("date")
    .resample("QS")
    .agg(
        no2_flaring_sum=("no2_flaring_sum", "mean"),
        no2_flaring_mean=("no2_flaring_mean", "mean"),
        no2_flaring_weighted=("no2_flaring_weighted", "mean"),
    )
    .reset_index()
)

# Log transforms (guard against log(0))
for col in ["no2_flaring_sum", "no2_flaring_mean", "no2_flaring_weighted"]:
    min_val = no2_flaring_quarterly[col].min()
    print(f"  {col}: min = {min_val:.2e}")
    no2_flaring_quarterly[f"log_{col}"] = np.log(
        no2_flaring_quarterly[col].clip(lower=1e-20)
    )

no2_flaring_quarterly.head()

  no2_flaring_sum: min = 1.06e-03
  no2_flaring_mean: min = 8.71e-06
  no2_flaring_weighted: min = 1.02e-05


,date,no2_flaring_sum,no2_flaring_mean,no2_flaring_weighted,log_no2_flaring_sum,log_no2_flaring_mean,log_no2_flaring_weighted
0,2012-01-01,0.001302,0.000011,0.000013,-6.643855,-11.447876,-11.280253
1,2012-04-01,0.001257,0.000010,0.000012,-6.679412,-11.483433,-11.296415
2,2012-07-01,0.001512,0.000012,0.000015,-6.494071,-11.298092,-11.114871
3,2012-10-01,0.001120,0.000009,0.000011,-6.794585,-11.598606,-11.445823
4,2013-01-01,0.001178,0.000010,0.000011,-6.743995,-11.548016,-11.387635


In [9]:
no2_flaring_annual = (
    no2_flaring_monthly.assign(year=lambda df: df["date"].dt.year)
    .groupby("year")
    .agg(
        no2_flaring_sum=("no2_trop", "sum"),
        no2_flaring_mean=("no2_trop", "mean"),
        total_flaring_sites=("flaring_site_count", "sum"),
        no2_weighted_num=(
            "no2_trop",
            lambda x: (
                x * no2_flaring_monthly.loc[x.index, "flaring_site_count"]
            ).sum(),
        ),
        no2_weighted_den=("flaring_site_count", "sum"),
    )
    .assign(
        no2_flaring_weighted=lambda df: df["no2_weighted_num"] / df["no2_weighted_den"]
    )
    .drop(columns=["no2_weighted_num", "no2_weighted_den"])
    .reset_index()
)

for col in ["no2_flaring_sum", "no2_flaring_mean", "no2_flaring_weighted"]:
    no2_flaring_annual[f"log_{col}"] = np.log(no2_flaring_annual[col].clip(lower=1e-20))

no2_flaring_annual.head()

,year,no2_flaring_sum,no2_flaring_mean,total_flaring_sites,no2_flaring_weighted,log_no2_flaring_sum,log_no2_flaring_mean,log_no2_flaring_weighted
0,2012,0.015572,0.000011,12420,0.000013,-4.162271,-11.451199,-11.277451
1,2013,0.015506,0.000011,12420,0.000012,-4.166533,-11.455461,-11.298501
2,2014,0.015896,0.000011,12420,0.000013,-4.141688,-11.430615,-11.277761
3,2015,0.018472,0.000013,12420,0.000015,-3.991499,-11.280427,-11.135543
4,2016,0.017362,0.000012,12420,0.000014,-4.053471,-11.342399,-11.204812


### Nighttime Lights (NTL) National Monthly

NTL serves as an independent proxy for economic activity. We use two variants:
- `ntl_sum`: total luminosity across Algeria — proxy for broad economic activity
- `ntl_gf_5km_sum`: luminosity within 5 km of gas flaring sites — proxy for hydrocarbon-specific activity

In [10]:
ntl_monthly = (
    pd.read_csv(DATA_DIR / "NTL" / "ntl_adm0_monthly.csv")
    .pipe(clean_names)
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .filter(["date", "ntl_sum", "ntl_gf_5km_sum", "ntl_gf_10km_sum"])
)

ntl_quarterly = ntl_monthly.set_index("date").resample("QS").mean().reset_index()

for col in ["ntl_sum", "ntl_gf_5km_sum", "ntl_gf_10km_sum"]:
    min_val = ntl_quarterly[col].min()
    print(f"  {col}: min = {min_val:.2e}")
    ntl_quarterly[f"log_{col}"] = np.log(ntl_quarterly[col].clip(lower=1e-20))

ntl_annual = (
    ntl_monthly.assign(year=lambda df: df["date"].dt.year)
    .groupby("year")
    .agg(ntl_sum=("ntl_sum", "mean"), ntl_gf_5km_sum=("ntl_gf_5km_sum", "mean"))
    .reset_index()
)
for col in ["ntl_sum", "ntl_gf_5km_sum"]:
    ntl_annual[f"log_{col}"] = np.log(ntl_annual[col].clip(lower=1e-20))

ntl_quarterly.head()

  ntl_sum: min = 2.23e+06
  ntl_gf_5km_sum: min = 7.51e+05
  ntl_gf_10km_sum: min = 8.41e+05


,date,ntl_sum,ntl_gf_5km_sum,ntl_gf_10km_sum,log_ntl_sum,log_ntl_gf_5km_sum,log_ntl_gf_10km_sum
0,2012-01-01,2.227140e+06,7.507462e+05,8.407120e+05,14.616229,13.528823,13.642004
1,2012-04-01,2.606200e+06,1.004217e+06,1.124268e+06,14.773404,13.819719,13.932643
2,2012-07-01,2.784620e+06,1.233267e+06,1.346382e+06,14.839622,14.025177,14.112931
3,2012-10-01,2.465038e+06,9.785253e+05,1.078685e+06,14.717718,13.793802,13.891253
4,2013-01-01,2.414655e+06,9.943465e+05,1.090425e+06,14.697067,13.809841,13.902078


### Quarterly Hydrocarbon Production

We load monthly oil and gas production from the "Hydro Mly" sheet to maximize sample size (~52 quarters for oil from 2012, ~44 for gas from 2014). Oil production (kb/d) is averaged to quarterly (daily rate); gas production (million std m³/month) is summed to quarterly totals with `min_count=3` to drop incomplete quarters. Forecasts beyond 2025 are excluded.

In [11]:
hydro_file = DATA_DIR / "gdp" / "Hydrocarbon DZA 2026.xlsx"

hydro_raw = pd.read_excel(hydro_file, sheet_name="Hydro Mly", header=None)

# Row 0 = monthly datetime timestamps as column headers
# Row 4 = Crude oil production (kb/d) — direct
# Row 12 = Natural gas production (million std m3/month)
h_dates = pd.to_datetime(hydro_raw.iloc[0, 1:])
oil_monthly = pd.to_numeric(hydro_raw.iloc[4, 1:], errors="coerce")
gas_monthly = pd.to_numeric(hydro_raw.iloc[12, 1:], errors="coerce")

hydro_monthly = (
    pd.DataFrame(
        {
            "date": h_dates.values,
            "oil_production": oil_monthly.values,
            "gas_production": gas_monthly.values,
        }
    )
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .query("date < '2026-01-01'")
    .set_index("date")
)

# Aggregate to quarterly: mean for oil (daily rate), sum for gas (monthly volume)
hydro_quarterly = (
    hydro_monthly.resample("QS")
    .agg(
        oil_production=("oil_production", "mean"),
        gas_production=pd.NamedAgg(
            column="gas_production", aggfunc=lambda x: x.sum(min_count=3)
        ),
    )
    .reset_index()
    .assign(
        log_oil=lambda df: np.log(df["oil_production"]),
        log_gas=lambda df: np.log(df["gas_production"]),
        quarter=lambda df: df["date"].dt.quarter.astype(str),
        year=lambda df: df["date"].dt.year,
    )
)

hydro_annual = (
    hydro_quarterly.dropna(subset=["oil_production"])
    .groupby("year")
    .agg(
        oil_production=("oil_production", "mean"),
        gas_production=("gas_production", "mean"),
    )
    .assign(
        log_oil=lambda df: np.log(df["oil_production"]),
        log_gas=lambda df: np.log(df["gas_production"]),
    )
    .reset_index()
)

print(
    f"Oil: {hydro_quarterly['oil_production'].notna().sum()} quarters, "
    f"{hydro_quarterly['date'].min():%Y-Q1} to {hydro_quarterly['date'].max():%Y}"
)
print(
    f"Gas: {hydro_quarterly['gas_production'].notna().sum()} quarters, "
    f"first valid {hydro_quarterly.dropna(subset=['gas_production'])['date'].min():%Y}"
)
hydro_quarterly.head()

Oil: 56 quarters, 2012-Q1 to 2025
Gas: 48 quarters, first valid 2014


,date,oil_production,gas_production,log_oil,log_gas,quarter,year
0,2012-01-01,1215.666667,NaN,7.103048,NaN,1,2012
1,2012-04-01,1226.666667,NaN,7.112056,NaN,2,2012
2,2012-07-01,1217.833333,NaN,7.104829,NaN,3,2012
3,2012-10-01,1197.333333,NaN,7.087852,NaN,4,2012
4,2013-01-01,1212.666667,NaN,7.100577,NaN,1,2013


### Quarterly Industrial Production Index (IPI)

In [12]:
ipi_file = DATA_DIR / "gdp" / "IPI & IPP 2026.xlsx"

ipi_raw = pd.read_excel(ipi_file, sheet_name="IPI Qly", header=None)

# Row 0: col 2+ has years (with header text in cols 0-1), Row 1: quarters
i_years = ipi_raw.iloc[0, 2:].ffill().values
i_quarters = ipi_raw.iloc[1, 2:].values

# Quarter labels change format: Q-1 (2013–2017) → Q1 (2018+)
quarter_map_ipi = {
    "Q-1": 1,
    "Q-2": 4,
    "Q-3": 7,
    "Q-4": 10,
    "Q1": 1,
    "Q2": 4,
    "Q3": 7,
    "Q4": 10,
}
i_dates = []
for y, q in zip(i_years, i_quarters):
    if pd.notna(y) and pd.notna(q) and str(q).strip() in quarter_map_ipi:
        i_dates.append(
            pd.Timestamp(year=int(y), month=quarter_map_ipi[str(q).strip()], day=1)
        )
    else:
        i_dates.append(pd.NaT)

i_data = ipi_raw.iloc[:, 2:]
i_data.columns = i_dates
i_labels = ipi_raw.iloc[:, 1]

# Key IPI sub-indices (0-indexed rows): 2=General, 3=Non-HC, 4=Manufacturing, 5=Energy, 6=Hydrocarbons, 7=Mines
IPI_ROWS = {
    2: "IPI General",
    3: "IPI Non-HC",
    4: "IPI Manufacturing",
    5: "IPI Energy",
    6: "IPI Hydrocarbons",
    7: "IPI Mines",
}

ipi_frames = []
for row_idx, label in IPI_ROWS.items():
    series = pd.to_numeric(i_data.iloc[row_idx], errors="coerce").dropna()
    temp = pd.DataFrame(
        {"date": series.index, "ipi": series.values, "ipi_sector": label}
    )
    ipi_frames.append(temp)

ipi_long = pd.concat(ipi_frames, ignore_index=True).assign(
    date=lambda df: pd.to_datetime(df["date"]), log_ipi=lambda df: np.log(df["ipi"])
)

ipi_long.head()

,date,ipi,ipi_sector,log_ipi
0,2013-01-01,87.8,IPI General,4.475062
1,2013-04-01,91.6,IPI General,4.517431
2,2013-07-01,93.9,IPI General,4.542230
3,2013-10-01,96.9,IPI General,4.573680
4,2014-01-01,89.8,IPI General,4.497585


### Data Preparation

In [13]:
df = (
    gdp_all.merge(no2_quarterly, on="date", how="inner", suffixes=("", "_no2"))
    .merge(
        no2_ind_quarterly[["date", "no2_trop_industrial", "log_no2_industrial"]],
        on="date",
        how="left",
    )
    .merge(no2_flaring_quarterly, on="date", how="left")
    .merge(ntl_quarterly, on="date", how="left")
    .sort_values(["sector", "date"])
    .assign(
        quarter=lambda df: df["date"].dt.quarter.astype("str"),
        gdp_yoy=lambda df: df.groupby("sector")["gdp"].pct_change(periods=4),
        gdp_qoq=lambda df: df.groupby("sector")["gdp"].pct_change(periods=1),
        no2_trop_yoy=lambda df: df.groupby("sector")["no2_trop"].pct_change(periods=4),
        no2_trop_qoq=lambda df: df.groupby("sector")["no2_trop"].pct_change(periods=1),
        no2_industrial_yoy=lambda df: df.groupby("sector")[
            "no2_trop_industrial"
        ].pct_change(periods=4),
        no2_industrial_qoq=lambda df: df.groupby("sector")[
            "no2_trop_industrial"
        ].pct_change(periods=1),
    )
)

df_detailed = df.query("sector in @DETAILED_SECTOR_ROWS.values()")
df_sectors = df.query("sector in @AGGREGATE_ROWS.values()")
df_summary = df.query("sector in @SUMMARY_ROWS.values()")

print(f"Merged dataframe: {len(df)} rows, {df.columns.tolist()}")
df.head()

Merged dataframe: 1944 rows, ['date', 'gdp', 'sector', 'gdp_log', 'quarter', 'no2_trop', 'no2_trop_total', 'log_no2', 'log_no2_total', 'quarter_no2', 'log_no2_lag_1', 'log_no2_lag_2', 'log_no2_lag_3', 'no2_trop_industrial', 'log_no2_industrial', 'no2_flaring_sum', 'no2_flaring_mean', 'no2_flaring_weighted', 'log_no2_flaring_sum', 'log_no2_flaring_mean', 'log_no2_flaring_weighted', 'ntl_sum', 'ntl_gf_5km_sum', 'ntl_gf_10km_sum', 'log_ntl_sum', 'log_ntl_gf_5km_sum', 'log_ntl_gf_10km_sum', 'gdp_yoy', 'gdp_qoq', 'no2_trop_yoy', 'no2_trop_qoq', 'no2_industrial_yoy', 'no2_industrial_qoq']


,date,gdp,sector,gdp_log,quarter,no2_trop,no2_trop_total,log_no2,log_no2_total,quarter_no2,...,ntl_gf_10km_sum,log_ntl_sum,log_ntl_gf_5km_sum,log_ntl_gf_10km_sum,gdp_yoy,gdp_qoq,no2_trop_yoy,no2_trop_qoq,no2_industrial_yoy,no2_industrial_qoq
1350,2012-01-01,535227.602555,Agriculture and fishing,13.190447,1,0.000007,0.000007,-11.882231,-11.921766,1,...,8.407120e+05,14.616229,13.528823,13.642004,NaN,NaN,NaN,NaN,NaN,NaN
1351,2012-04-01,535130.268018,Agriculture and fishing,13.190265,2,0.000007,0.000007,-11.863194,-11.920607,2,...,1.124268e+06,14.773404,13.819719,13.932643,NaN,-0.000182,NaN,0.015644,NaN,-0.177690
1352,2012-07-01,460552.165069,Agriculture and fishing,13.040181,3,0.000008,0.000008,-11.730008,-11.761442,3,...,1.346382e+06,14.839622,14.025177,14.112931,NaN,-0.139364,NaN,0.145621,NaN,0.274382
1353,2012-10-01,541990.810212,Agriculture and fishing,13.203004,4,0.000007,0.000006,-11.943599,-11.983659,4,...,1.078685e+06,14.717718,13.793802,13.891253,NaN,0.176828,NaN,-0.193346,NaN,-0.130100
1354,2013-01-01,604807.190887,Agriculture and fishing,13.312665,1,0.000007,0.000006,-11.925341,-11.948862,1,...,1.090425e+06,14.697067,13.809841,13.902078,0.13,0.115899,-0.045399,0.017073,-0.218375,-0.142580


## Exploratory Data Analysis

In [14]:
nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date"], empty=False
)

base = alt.Chart(no2_national).encode(x="date:T")

line = base.mark_line().encode(
    y=alt.Y(
        "no2_trop:Q", title="Tropospheric NO₂ (mol/m²)", scale=alt.Scale(zero=False)
    ),
)

loess = (
    base.mark_line(color="#FF9800", strokeWidth=2)
    .transform_loess("date", "no2_trop", bandwidth=0.3)
    .encode(
        y=alt.Y("no2_trop:Q"),
    )
)

rule = (
    base.mark_rule(color="gray")
    .encode(
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-%m"),
            alt.Tooltip("no2_trop:Q", title="NO₂", format=".2e"),
        ],
    )
    .add_params(nearest)
)


alt.layer(line, loess, rule).properties(
    width=700,
    height=300,
    title=alt.Title(
        "Monthly Tropospheric NO₂ at National Level",
        subtitle="Orange line represents LOESS smoothed trend",
    ),
)

alt.LayerChart(...)

In [15]:
nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date"], empty=False
)

base = alt.Chart(no2_quarterly).encode(x="date:T")

line = base.mark_line().encode(
    y=alt.Y(
        "no2_trop:Q", title="Tropospheric NO₂ (mol/m²)", scale=alt.Scale(zero=False)
    ),
)

loess = (
    base.mark_line(color="#FF9800", strokeWidth=2)
    .transform_loess("date", "no2_trop", bandwidth=0.3)
    .encode(
        y=alt.Y("no2_trop:Q"),
    )
)

rule = (
    base.mark_rule(color="gray")
    .encode(
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-%m"),
            alt.Tooltip("no2_trop:Q", title="NO₂", format=".2e"),
        ],
    )
    .add_params(nearest)
)
(
    alt.layer(line, loess, rule).properties(
        width=700,
        height=300,
        title=alt.Title(
            "Quarterly Tropospheric NO₂ at National Level",
            subtitle="Orange line represents LOESS smoothed trend",
        ),
    )
)

alt.LayerChart(...)

In [16]:
alt.Chart(df_summary).mark_line().encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("gdp:Q", title="", scale=alt.Scale(zero=False)),
    color=alt.Color("sector:N", title="Sector"),
    tooltip=[
        alt.Tooltip("sector:N", title="Sector"),
        alt.Tooltip("date:T", title="Date", format="%Y-Q%q"),
        alt.Tooltip("gdp:Q", title="GDP (constant prices, bn DZD)", format=",.0f"),
    ],
).properties(
    width=700,
    height=300,
    title=alt.Title(
        "Quarterly GDP at National Level",
        subtitle="",
    ),
)

alt.Chart(...)

In [17]:
(
    alt.Chart(df_sectors.query('date >= "2012-01-01"'))
    .mark_line()
    .encode(
        x=alt.X("date:T", title="Date"),
        y=alt.Y("gdp:Q", title="", scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-Q%q"),
            alt.Tooltip("gdp:Q", title="GDP (constant prices, bn DZD)", format=",.0f"),
        ],
    )
    .properties(width=250, height=150)
    .facet(
        facet=alt.Facet("sector:N", title=None),
        columns=3,
    )
    .properties(title="Quarterly GDP by Aggregate Sector")
)

alt.FacetChart(...)

In [18]:
gdp_melt = (
    df.copy()
    .query("sector == 'Total GDP'")
    .filter(["date", "gdp", "no2_trop"])
    .dropna()
    .assign(
        gdp_z=lambda df: standardize_series(df["gdp"]),
        no2_z=lambda df: standardize_series(df["no2_trop"]),
    )
    .melt(
        id_vars="date",
        value_vars=["gdp_z", "no2_z"],
        var_name="variable",
        value_name="value",
    )
    .assign(
        variable=lambda df: df["variable"].map(
            {"gdp_z": "Total GDP", "no2_z": "NO₂ (national)"}
        )
    )
)

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date"], empty=False
)

base = alt.Chart(gdp_melt).encode(x=alt.X("date:T", title="Date"))

lines = base.mark_line().encode(
    y=alt.Y("value:Q", title="Standardized Value (z-score)"),
    color=alt.Color("variable:N", title=None),
)

rules = (
    base.transform_pivot("variable", value="value", groupby=["date"])
    .mark_rule(color="gray")
    .encode(
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", format="%Y-Q%q"),
            alt.Tooltip("Total GDP:Q", format=".2f"),
            alt.Tooltip("NO₂ (national):Q", format=".2f"),
        ],
    )
    .add_params(nearest)
)


alt.layer(lines, rules).properties(
    width=700, height=300, title="Standardized Total GDP vs National NO₂"
)

alt.LayerChart(...)

In [19]:
sector_melt = (
    (
        df.copy()
        .query("sector in @AGGREGATE_ROWS.values()")
        .assign(
            gdp_z=lambda df: df.groupby("sector")["gdp"].transform(standardize_series),
            no2_z=lambda df: df.groupby("sector")["no2_trop"].transform(
                standardize_series
            ),
        )
    )
    .melt(
        id_vars=["date", "sector"],
        value_vars=["gdp_z", "no2_z"],
        var_name="variable",
        value_name="value",
    )
    .assign(variable=lambda df: df["variable"].map({"gdp_z": "GDP", "no2_z": "NO₂"}))
)

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "sector"], empty=False
)

base = alt.Chart(sector_melt).encode(x=alt.X("date:T", title=""))

lines = base.mark_line().encode(
    y=alt.Y("value:Q", title=""),
    color=alt.Color("variable:N", title=None),
)

rules = (
    base.transform_pivot("variable", value="value", groupby=["date", "sector"])
    .mark_rule(color="gray")
    .encode(
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", format="%Y-Q%q"),
            alt.Tooltip("GDP:Q", title="Standardized GDP", format=".2f"),
            alt.Tooltip("NO₂:Q", title="Standardized NO₂", format=".2f"),
        ],
    )
    .add_params(nearest)
)

alt.layer(lines, rules).properties(width=250, height=150).facet(
    facet=alt.Facet("sector:N", title=None),
    columns=3,
).properties(title="Standardized GDP and NO₂ by Sector")

alt.FacetChart(...)

In [20]:
no2_ts = no2_national.set_index("date")["no2_trop"].dropna()
no2_ts = no2_ts.asfreq("MS")

stl = STL(no2_ts, period=12, robust=True)
stl_result = stl.fit()

stl_df = pd.DataFrame(
    {
        "date": no2_ts.index,
        "Observed": stl_result.observed,
        "Trend": stl_result.trend,
        "Seasonal": stl_result.seasonal,
        "Residual": stl_result.resid,
    }
).melt(id_vars="date", var_name="component", value_name="value")

alt.Chart(stl_df).mark_line().encode(
    x=alt.X("date:T", title="Date"),
    y=alt.Y("value:Q", title=None, scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("date:T", format="%Y-%m"),
        alt.Tooltip("value:Q", format=".2e"),
    ],
).properties(width=700, height=120).facet(
    row=alt.Row(
        "component:N", title=None, sort=["Observed", "Trend", "Seasonal", "Residual"]
    ),
).properties(title="STL Decomposition of Monthly NO₂")

alt.FacetChart(...)

In [21]:
month_order = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

alt.Chart(no2_national).mark_boxplot(extent="min-max").encode(
    x=alt.X("month:N", title="Month", sort=month_order),
    y=alt.Y("no2_trop:Q", title="Tropospheric NO₂", scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("month:N", title="Month"),
        alt.Tooltip("no2_trop:Q", title="Tropospheric NO₂", format=".2e"),
    ],
).properties(width=600, height=300, title="Seasonal Distribution of Monthly NO₂")

alt.Chart(...)

In [22]:
TARGET_SECTORS = [
    "Hydrocarbons",
    "Non-HC manufacturing",
    "Construction",
    "Transport and communications",
    "Low VA services",
    "Oil and gas extraction and related services",
    "Refining and coking",
]

df_target = df.copy().query("sector in @TARGET_SECTORS")

target_melt = (
    df_target.assign(
        gdp_z=lambda d: d.groupby("sector")["gdp"].transform(standardize_series),
        no2_z=lambda d: d.groupby("sector")["no2_trop"].transform(standardize_series),
    )
    .melt(
        id_vars=["date", "sector"],
        value_vars=["gdp_z", "no2_z"],
        var_name="variable",
        value_name="value",
    )
    .assign(variable=lambda d: d["variable"].map({"gdp_z": "GDP", "no2_z": "NO₂"}))
)

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "sector"], empty=False
)

base = alt.Chart(target_melt).encode(x=alt.X("date:T", title="Date"))

lines = base.mark_line().encode(
    y=alt.Y("value:Q", title="Standardized Value (z-score)"),
    color=alt.Color("variable:N", title=None),
)

rules = (
    base.transform_pivot("variable", value="value", groupby=["date", "sector"])
    .mark_rule(color="gray")
    .encode(
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("GDP:Q", title="Standardized GDP", format=".2f"),
            alt.Tooltip("NO₂:Q", title="Standardized NO₂", format=".2f"),
        ],
    )
    .add_params(nearest)
)

alt.layer(lines, rules).properties(width=280, height=160).facet(
    facet=alt.Facet("sector:N", title=None, sort=TARGET_SECTORS),
    columns=3,
).properties(title="Standardized GDP and NO₂ for Target Sectors")

alt.FacetChart(...)

## NO2-GDP Models with Nighttime Lights (NTL)

### Hydrocarbon Model

We focus first on the hydrocarbon sector and run regression using mean NO₂ and sum of nighttime lights near flaring locations. We include both without and with quarter dummies to control for seasonality.

* No quarter dummies: `log(GDP) ~ log(ntl_gf_5km_sum) + log(NO₂_flaring_mean)`
* With quarter dummies: `log(GDP) ~ log(ntl_gf_5km_sum) + log(NO₂_flaring_mean) + quarter`

In [23]:
hc_quarterly = (
    df.query("sector == 'Hydrocarbons'")
    .dropna(subset=["log_ntl_gf_10km_sum", "log_no2_flaring_mean"])
    .copy()
)

hc_model = smf.ols(
    "gdp_log ~ log_ntl_gf_10km_sum + log_no2_flaring_mean", data=hc_quarterly
).fit()
hc_model_seasonal = smf.ols(
    "gdp_log ~ log_ntl_gf_10km_sum + log_no2_flaring_mean + C(quarter)",
    data=hc_quarterly,
).fit()


star_hc = Stargazer([hc_model, hc_model_seasonal])
star_hc.covariate_order(["Intercept", "log_ntl_gf_10km_sum", "log_no2_flaring_mean"])
star_hc.custom_columns(["Without Quarter Dummies", "With Quarter Dummies"], [1, 1])
star_hc.rename_covariates(
    {
        "log_ntl_gf_10km_sum": "Log NTL in flaring locations",
        "log_no2_flaring_mean": "Log NO₂ in flaring locations",
    }
)
star_hc

### Oil & Gas Production

In [24]:
# Merge hydrocarbon production with satellite data
hydro_sat = (
    hydro_quarterly.merge(
        ntl_quarterly[["date", "log_ntl_gf_10km_sum"]], on="date", how="inner"
    )
    .merge(
        no2_flaring_quarterly[["date", "log_no2_flaring_mean"]], on="date", how="inner"
    )
    .dropna(subset=["log_oil", "log_ntl_gf_10km_sum", "log_no2_flaring_mean"])
)

print(
    f"hydro_sat: {len(hydro_sat)} rows, {hydro_sat['date'].min():%Y-Q1} to {hydro_sat['date'].max():%Y-Q4}"
)
print(
    f"  Oil valid: {hydro_sat['log_oil'].notna().sum()}, Gas valid: {hydro_sat['log_gas'].notna().sum()}"
)
hydro_sat.head()

hydro_sat: 56 rows, 2012-Q1 to 2025-Q4
  Oil valid: 56, Gas valid: 48


,date,oil_production,gas_production,log_oil,log_gas,quarter,year,log_ntl_gf_10km_sum,log_no2_flaring_mean
0,2012-01-01,1215.666667,NaN,7.103048,NaN,1,2012,13.642004,-11.447876
1,2012-04-01,1226.666667,NaN,7.112056,NaN,2,2012,13.932643,-11.483433
2,2012-07-01,1217.833333,NaN,7.104829,NaN,3,2012,14.112931,-11.298092
3,2012-10-01,1197.333333,NaN,7.087852,NaN,4,2012,13.891253,-11.598606
4,2013-01-01,1212.666667,NaN,7.100577,NaN,1,2013,13.902078,-11.548016


In [25]:
oil_models = []
oil_formulas = [
    "log_oil ~ log_ntl_gf_10km_sum",
    "log_oil ~ log_ntl_gf_10km_sum + C(quarter)",
    "log_oil ~ log_no2_flaring_mean",
    "log_oil ~ log_no2_flaring_mean + C(quarter)",
    "log_oil ~ log_ntl_gf_10km_sum + log_no2_flaring_mean",
    "log_oil ~ log_ntl_gf_10km_sum + log_no2_flaring_mean + C(quarter)",
]

for formula in oil_formulas:
    oil_models.append(smf.ols(formula, data=hydro_sat).fit())

star_oil = Stargazer(oil_models)
star_oil.covariate_order(["Intercept", "log_ntl_gf_10km_sum", "log_no2_flaring_mean"])
star_oil.custom_columns(
    ["O1", "O2", "O3", "O4", "O5", "O6"],
    [1, 1, 1, 1, 1, 1],
)
star_oil.rename_covariates(
    {
        "log_ntl_gf_10km_sum": "Log NTL (flaring 10km)",
        "log_no2_flaring_mean": "Log NO₂ (flaring grids)",
    }
)
star_oil.title("Oil Production (kb/d) ~ Satellite Proxies")
star_oil

In [26]:
hydro_sat_gas = hydro_sat.dropna(subset=["log_gas"])

gas_models = []
gas_formulas = [
    "log_gas ~ log_ntl_gf_10km_sum",
    "log_gas ~ log_ntl_gf_10km_sum + C(quarter)",
    "log_gas ~ log_no2_flaring_mean",
    "log_gas ~ log_no2_flaring_mean + C(quarter)",
    "log_gas ~ log_ntl_gf_10km_sum + log_no2_flaring_mean",
    "log_gas ~ log_ntl_gf_10km_sum + log_no2_flaring_mean + C(quarter)",
]

for formula in gas_formulas:
    gas_models.append(smf.ols(formula, data=hydro_sat_gas).fit())

star_gas = Stargazer(gas_models)
star_gas.covariate_order(["Intercept", "log_ntl_gf_10km_sum", "log_no2_flaring_mean"])
star_gas.custom_columns(
    ["G1", "G2", "G3", "G4", "G5", "G6"],
    [1, 1, 1, 1, 1, 1],
)
star_gas.rename_covariates(
    {
        "log_ntl_gf_10km_sum": "Log NTL (flaring 10km)",
        "log_no2_flaring_mean": "Log NO₂ (flaring grids)",
    }
)
star_gas.title("Gas Production (million std m³) ~ Satellite Proxies")
star_gas

### Sector Models

For each of the 5 target sectors, we estimate two models:
* National Models: `log(GDP) ~ log(NTL) + log(NO₂_national) + quarter`
* Flaring models: `log(GDP) ~ log(NTL) + log(NO₂_flaring_mean) + quarter`

### NTL + National NO₂

In [27]:
SECTOR_NTL = {
    "Hydrocarbons": "log_ntl_sum",
    # "Hydrocarbons": "log_ntl_gf_5km_sum",
    "Non-HC manufacturing": "log_ntl_sum",
    "Construction": "log_ntl_sum",
    "Transport and communications": "log_ntl_sum",
    "Low VA services": "log_ntl_sum",
    "Oil and gas extraction and related services": "log_ntl_sum",
    "Refining and coking": "log_ntl_sum",
}

results_national = []
for sector in TARGET_SECTORS:
    ntl_var = SECTOR_NTL[sector]
    sector_df = df.query("sector == @sector").dropna(
        subset=["gdp_log", ntl_var, "log_no2"]
    )
    formula = f"gdp_log ~ {ntl_var} + log_no2 + C(quarter)"
    mod = smf.ols(formula, data=sector_df).fit()
    results_national.append((sector, mod))

star_national = Stargazer([m for _, m in results_national])
star_national.custom_columns(
    [s for s, _ in results_national], [1] * len(results_national)
)
star_national.covariate_order(["Intercept", "log_ntl_sum", "log_no2"])
star_national.rename_covariates(
    {
        "log_ntl_sum": "Log NTL (national)",
        "log_no2": "Log NO₂ (national)",
    }
)
star_national

In [28]:
plot_coefficients(
    results_national,
    var="log_no2",
    title="NO₂-GDP Elasticity by Sector (National NO₂)",
    subtitle="With seasonality controls",
)

alt.LayerChart(...)

### NTL + Flaring NO₂

In [29]:
results_flaring = []
for sector in TARGET_SECTORS:
    ntl_var = SECTOR_NTL[sector]
    sector_df = df.query("sector == @sector").dropna(
        subset=["gdp_log", ntl_var, "log_no2_flaring_mean"]
    )
    formula = f"gdp_log ~ {ntl_var} + log_no2_flaring_mean + C(quarter)"
    mod = smf.ols(formula, data=sector_df).fit()
    results_flaring.append((sector, mod))

star_flaring = Stargazer([m for _, m in results_flaring])
star_flaring.custom_columns([s for s, _ in results_flaring], [1] * len(results_flaring))
star_flaring.covariate_order(["Intercept", "log_ntl_sum", "log_no2_flaring_mean"])
star_flaring.rename_covariates(
    {
        "log_ntl_sum": "Log NTL (national)",
        "log_no2_flaring_mean": "Log NO₂ in flaring locations",
    }
)
star_flaring

In [30]:
plot_coefficients(
    results_flaring,
    var="log_no2_flaring_mean",
    title="NO₂-GDP Elasticity by Sector (NO₂ in Flaring Locations)",
    subtitle="With seasonality controls",
)

alt.LayerChart(...)

## Bivariate NO₂-GDP Models

### National Level

In [31]:
SELECTED_SECTORS = list(AGGREGATE_ROWS.values()) + [
    "Hydrocarbons",
    "Oil and gas extraction and related services",
    "Refining and coking",
]

results_L0 = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), "log_no2", "gdp_log"
)

models_L0 = [m for _, m in results_L0]
labels_L0 = [s for s, _ in results_L0]
star_L0 = Stargazer(models_L0)
star_L0.custom_columns(labels_L0, [1] * len(labels_L0))
star_L0.covariate_order(["Intercept", "log_no2"])
star_L0.rename_covariates({"log_no2": "Log NO₂ (national)"})
star_L0

In [32]:
plot_coefficients(
    results_L0,
    var="log_no2",
    title="NO₂-GDP Elasticity at National Level across Sectors at No Lag",
    subtitle="Without seasonality controls",
)

alt.LayerChart(...)

In [33]:
results_seasonal = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), ["log_no2", "quarter"]
)

star_seasonal = Stargazer([m for _, m in results_seasonal])
star_seasonal.custom_columns(
    [s for s, _ in results_seasonal], [1] * len(results_seasonal)
)
star_seasonal.covariate_order(["Intercept", "log_no2"])
star_seasonal.rename_covariates({"log_no2": "Log NO₂ (national)"})
star_seasonal

In [34]:
plot_coefficients(
    results_seasonal,
    var=["log_no2"],
    title="NO₂-GDP Elasticity at National Level across Sectors at No Lag",
    subtitle="With seasonality controls",
)

alt.LayerChart(...)

In [35]:
results_L1 = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), ["log_no2_lag_1"], "gdp_log"
)

models_L1 = [m for _, m in results_L1]
labels_L1 = [s for s, _ in results_L1]
star_L1 = Stargazer(models_L1)
star_L1.custom_columns(labels_L1, [1] * len(labels_L1))
star_L1.covariate_order(["Intercept", "log_no2_lag_1"])
star_L1.rename_covariates({"log_no2_lag_1": "Log NO₂ (national, lag 1 quarter)"})
star_L1

In [36]:
plot_coefficients(
    results_L1,
    var="log_no2_lag_1",
    title="NO₂-GDP Elasticity at National Level across Sectors at Lagged 1 quarter",
    subtitle="Without seasonality controls",
)

alt.LayerChart(...)

In [37]:
results_L2 = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), ["log_no2_lag_2"], "gdp_log"
)

models_L2 = [m for _, m in results_L2]
labels_L2 = [s for s, _ in results_L2]
star_L2 = Stargazer(models_L2)
star_L2.custom_columns(labels_L2, [1] * len(labels_L2))
star_L2.covariate_order(["Intercept", "log_no2_lag_2"])
star_L2.rename_covariates({"log_no2_lag_2": "Log NO₂ (national, lag 2 quarters)"})
star_L2

In [38]:
plot_coefficients(
    results_L2,
    var="log_no2_lag_2",
    title="NO₂-GDP Elasticity at National Level across Sectors at Lagged 2 quarters",
    subtitle="Without seasonality controls",
)

alt.LayerChart(...)

In [39]:
results_L3 = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), "log_no2_lag_3", "gdp_log"
)

models_L3 = [m for _, m in results_L3]
labels_L3 = [s for s, _ in results_L3]
star_L3 = Stargazer(models_L3)
star_L3.custom_columns(labels_L3, [1] * len(labels_L3))
star_L3.covariate_order(["Intercept", "log_no2_lag_3"])
star_L3.rename_covariates({"log_no2_lag_3": "Log NO₂ (national, lag 3 quarters)"})
star_L3

In [40]:
plot_coefficients(
    results_L3,
    var="log_no2_lag_3",
    title="NO₂-GDP Elasticity at National Level across Sectors at Lagged 3 quarters",
    subtitle="Without seasonality controls",
)

alt.LayerChart(...)

In [41]:
vif_data = (
    no2_quarterly.copy()
    .filter(["log_no2", "log_no2_lag_1", "log_no2_lag_2", "log_no2_lag_3"])
    .dropna()
)

vif_results = pd.DataFrame(
    {
        "Variable": vif_data.columns,
        "VIF": [
            variance_inflation_factor(vif_data.values, i)
            for i in range(vif_data.shape[1])
        ],
    }
)
print("VIF for NO₂ predictors (contemporaneous + lags):")
print("VIF > 10 indicates problematic multicollinearity")
vif_results

VIF for NO₂ predictors (contemporaneous + lags):
VIF > 10 indicates problematic multicollinearity


,Variable,VIF
0,log_no2,13060.344484
1,log_no2_lag_1,13030.213241
2,log_no2_lag_2,13304.486079
3,log_no2_lag_3,13438.408280


## Change Models

In [42]:
results_yoy = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), "no2_trop_yoy", "gdp_yoy"
)
star_yoy = Stargazer([m for _, m in results_yoy])
star_yoy.custom_columns([s for s, _ in results_yoy], [1] * len(results_yoy))
star_yoy.covariate_order(["Intercept", "no2_trop_yoy"])
star_yoy.rename_covariates({"no2_trop_yoy": "NO₂ YoY Growth"})
star_yoy

In [43]:
plot_coefficients(
    results_yoy,
    var="no2_trop_yoy",
    title="YoY GDP Growth ~ YoY NO₂ Growth",
    subtitle="",
)

alt.LayerChart(...)

In [44]:
results_qoq = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), "no2_trop_qoq", "gdp_qoq"
)
star_qoq = Stargazer([m for _, m in results_qoq])
star_qoq.custom_columns([s for s, _ in results_qoq], [1] * len(results_qoq))
star_qoq.covariate_order(["Intercept", "no2_trop_qoq"])
star_qoq.rename_covariates({"no2_trop_qoq": "NO₂ QoQ Growth"})
star_qoq

In [45]:
plot_coefficients(
    results_qoq,
    var="no2_trop_qoq",
    title="QoQ GDP Change ~ QoQ NO₂ Change",
    subtitle="",
)

alt.LayerChart(...)

## Industrial Wilayas Level

This section repeats the core models using NO₂ averaged only over the four industrial wilayas: Alger, Oran, Ouargla, and Skikda.

In [46]:
no2_selected = no2_adm1[no2_adm1["name_1"].isin(INDUSTRIAL_WILAYAS)].copy()

alt.Chart(no2_selected).mark_line().encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("no2_trop:Q", title="Tropospheric NO₂", scale=alt.Scale(zero=False)),
    tooltip=[
        "name_1:N",
        alt.Tooltip("date:T", format="%Y-%m"),
        alt.Tooltip("no2_trop:Q", format=".2e"),
    ],
).properties(width=300, height=150).facet(
    facet=alt.Facet("name_1:N", title=""),
    columns=2,
).properties(title="Tropospheric NO₂ in Industrial Wilayas")

alt.FacetChart(...)

In [47]:
results_industrial = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"), "log_no2_industrial", "gdp_log"
)

star_ind = Stargazer([m for _, m in results_industrial])
star_ind.custom_columns(
    [s for s, _ in results_industrial], [1] * len(results_industrial)
)
star_ind.covariate_order(["Intercept", "log_no2_industrial"])
star_ind.rename_covariates({"log_no2_industrial": "Log NO₂ (industrial wilayas)"})
star_ind

In [48]:
plot_coefficients(
    results_industrial,
    var="log_no2_industrial",
    title="NO₂-GDP Elasticity with NO₂ Industrial Wilayas Only",
    subtitle="Without seasonality controls",
)

alt.LayerChart(...)

In [49]:
results_industrial_seasonal = fit_sector_models(
    df.query("sector in @SELECTED_SECTORS"),
    ["log_no2_industrial", "quarter"],
    "gdp_log",
)

star_ind_seasonal = Stargazer([m for _, m in results_industrial_seasonal])
star_ind_seasonal.custom_columns(
    [s for s, _ in results_industrial_seasonal], [1] * len(results_industrial_seasonal)
)
star_ind_seasonal.covariate_order(["Intercept", "log_no2_industrial"])
star_ind_seasonal.rename_covariates(
    {"log_no2_industrial": "Log NO₂ (industrial wilayas)"}
)
star_ind_seasonal

In [50]:
plot_coefficients(
    results_industrial_seasonal,
    var="log_no2_industrial",
    title="NO₂-GDP Elasticity with NO₂ Industrial Wilayas Only",
    subtitle="With seasonality controls",
)

alt.LayerChart(...)